<a href="https://colab.research.google.com/github/arjunnamburi/ml_death_overs_ipl/blob/main/analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from google.colab import drive

drive.mount('/content/drive')
DATA = '/content/drive/MyDrive/IPL PROJECT ALL DOCS/data/'
SEED = 42                      # fixed seed, so the bootstrap results are reproducible

Mounted at /content/drive


In [2]:
# paste your Drive mount and DATA = '...' lines from the top of models.ipynb here
import numpy as np, pandas as pd
B = pd.read_parquet(DATA + 'bundle_2025.parquet')
y25 = B['y'].to_numpy().astype(int)
P_x25, P_l25_cal, P_b1 = (B[[f'p{t}_{k}' for k in range(7)]].to_numpy() for t in ('x', 'l', 'b1'))
i = np.arange(len(y25))
LX, LL, LB = (-np.log(P[i, y25]) for P in (P_x25, P_l25_cal, P_b1))
match = B['match_id'].to_numpy()
print(len(B), LX.mean().round(4), LL.mean().round(4), LB.mean().round(4))   # expect 15809 1.4727 1.4762 1.5119

RUNS = np.array([0, 1, 2, 3, 4, 6, 0], float)
runs_act, runs_exp = RUNS[y25], P_x25 @ RUNS
six_act, six_exp = (y25 == 5).astype(float), P_x25[:, 5]
wkt_act, wkt_exp = (y25 == 6).astype(float), P_x25[:, 6]
bat, bowl = B['batter'].to_numpy(), B['bowler'].to_numpy()
print('runs per ball, expected vs actual:', runs_exp.mean().round(3), runs_act.mean().round(3))

15809 1.4727 1.4762 1.5119
runs per ball, expected vs actual: 1.615 1.554


In [3]:
def slice_paired(la, lb, mask, match, n_boot=2000, z_bonf=3.254, seed=0):
    """Mean of (la - lb) over the slice, 95% match-bootstrap interval, SE, and the Bonferroni flag."""
    diff, m = (la - lb)[mask], match[mask]
    codes, _ = pd.factorize(m)                          # match -> 0..K-1
    s, c = np.bincount(codes, weights=diff), np.bincount(codes)   # per-match sum of differences and ball count
    idx = np.random.default_rng(seed).integers(0, len(s), size=(n_boot, len(s)))   # resample matches with replacement
    boot = s[idx].sum(axis=1) / c[idx].sum(axis=1)      # average difference in each fake season
    mean, se = diff.mean(), boot.std()
    low, high = np.percentile(boot, [2.5, 97.5])
    return mean, low, high, se, abs(mean) > z_bonf * se

In [4]:
def player_table(who, min_balls):
    """Per player with >= min_balls 2025 balls: (A) X-hist - B1, (B) X-hist log-loss, (C) actual - expected."""
    rows = []
    for p in pd.unique(who):
        m = who == p
        n = int(m.sum())
        if n < min_balls:
            continue
        A = slice_paired(LX, LB, m, match)                   # per-ball gain over B1 (negative = model reads them better)
        R = slice_paired(runs_act, runs_exp, m, match)       # runs per ball above X-hist's expectation
        S = slice_paired(six_act, six_exp, m, match)
        W = slice_paired(wkt_act, wkt_exp, m, match)
        rows.append({'player': p, 'balls': n, 'matches': len(np.unique(match[m])),
                     'gain vs B1': A[0], 'gain low': A[1], 'gain high': A[2],
                     'LL X-hist': LX[m].mean(),
                     'SR above exp': 100 * R[0], 'SR low': 100 * R[1], 'SR high': 100 * R[2],
                     'runs above exp': n * R[0], 'sixes above exp': n * S[0],
                     'wkts above exp': n * W[0], 'wkts low': n * W[1], 'wkts high': n * W[2]})
    return pd.DataFrame(rows).set_index('player')

In [5]:
BAT, BOWL = player_table(bat, 150), player_table(bowl, 120)
print(len(BAT), 'batters and', len(BOWL), 'bowlers qualify')
def top(df, col, asc, cols, title, n=10):
    print(title); display(df.sort_values(col, ascending=asc).head(n)[['balls'] + cols].round(3))
A_cols = ['gain vs B1', 'gain low', 'gain high']
top(BAT, 'gain vs B1', True,  A_cols, '(A) Batters the model reads best (gain vs B1 most negative)')
top(BAT, 'gain vs B1', False, A_cols, '(A) Batters the model reads least')
top(BAT, 'LL X-hist', True,  ['LL X-hist'], '(B) Most predictable batters (curiosity)')
top(BAT, 'LL X-hist', False, ['LL X-hist'], '(B) Least predictable batters (curiosity)')
C_bat = ['SR above exp', 'SR low', 'SR high', 'runs above exp', 'sixes above exp']
top(BAT, 'runs above exp', False, C_bat, '(C) Batters who beat the model most (runs above expected)')
top(BAT, 'runs above exp', True,  C_bat, '(C) Batters the model over-rated most')
top(BOWL, 'gain vs B1', True,  A_cols, '(A) Bowlers the model reads best')
top(BOWL, 'gain vs B1', False, A_cols, '(A) Bowlers the model reads least')
C_bowl = ['runs above exp', 'SR above exp', 'wkts above exp', 'wkts low', 'wkts high']
top(BOWL, 'runs above exp', True,  C_bowl, '(C) Bowlers who conceded least vs expected')
top(BOWL, 'wkts above exp', False, C_bowl, '(C) Bowlers with most wickets above expected')
BAT.to_csv(DATA + 'players_bat_2025.csv'); BOWL.to_csv(DATA + 'players_bowl_2025.csv')

41 batters and 51 bowlers qualify
(A) Batters the model reads best (gain vs B1 most negative)


,balls,gain vs B1,gain low,gain high
player,,,,
Priyansh Arya,218,-0.088,-0.148,-0.022
Abhishek Sharma,225,-0.084,-0.178,0.002
N Pooran,265,-0.081,-0.143,-0.017
SS Iyer,317,-0.068,-0.100,-0.030
PD Salt,223,-0.064,-0.116,-0.013
R Parag,235,-0.064,-0.119,-0.014
H Klaasen,279,-0.063,-0.093,-0.033
Shubman Gill,370,-0.053,-0.081,-0.026
JC Buttler,303,-0.052,-0.072,-0.030


(A) Batters the model reads least


,balls,gain vs B1,gain low,gain high
player,,,,
Tilak Varma,238,0.029,0.004,0.052
A Raghuvanshi,214,0.022,-0.005,0.048
Nithish Kumar Reddy,152,0.012,-0.041,0.045
RR Pant,202,0.012,-0.018,0.036
F du Plessis,153,0.009,-0.031,0.046
SE Rutherford,169,-0.009,-0.041,0.022
SO Hetmyer,164,-0.011,-0.054,0.026
RM Patidar,199,-0.014,-0.064,0.029
Abishek Porel,195,-0.015,-0.049,0.013


(B) Most predictable batters (curiosity)


...

,balls,gain vs B1,gain low,gain high
player,,,,
Suyash Sharma,282,0.003,-0.037,0.039
Prince Yadav,137,0.003,-0.033,0.038
Kuldeep Yadav,306,-0.004,-0.028,0.019
Mohammed Siraj,324,-0.005,-0.035,0.024
M Prasidh Krishna,330,-0.007,-0.050,0.039
JJ Bumrah,260,-0.008,-0.038,0.020
HV Patel,239,-0.012,-0.054,0.034
DS Rathi,312,-0.013,-0.050,0.023
A Kamboj,129,-0.014,-0.047,0.017


(C) Bowlers who conceded least vs expected


,balls,runs above exp,SR above exp,wkts above exp,wkts low,wkts high
player,,,,,,
DS Rathi,312,-99.405,-31.861,-2.841,-8.341,2.993
Kuldeep Yadav,306,-95.924,-31.348,-0.154,-7.039,7.067
M Prasidh Krishna,330,-95.136,-28.829,3.436,-3.888,11.337
JJ Bumrah,260,-82.987,-31.918,0.719,-9.022,10.983
Arshdeep Singh,332,-66.927,-20.159,-3.541,-12.476,4.601
Noor Ahmad,300,-62.423,-20.808,7.369,-1.208,16.412
HV Patel,239,-55.814,-23.353,1.059,-7.138,9.197
Sandeep Sharma,219,-46.827,-21.382,-3.956,-6.846,-0.875
CV Varun,276,-45.395,-16.448,1.737,-5.351,8.494


(C) Bowlers with most wickets above expected


,balls,runs above exp,SR above exp,wkts above exp,wkts low,wkts high
player,,,,,,
Noor Ahmad,300,-62.423,-20.808,7.369,-1.208,16.412
E Malinga,136,-28.070,-20.640,5.073,3.033,7.209
KH Pandya,270,-35.318,-13.081,3.990,-2.582,10.246
R Sai Kishore,231,-28.878,-12.501,3.876,-1.052,8.392
JR Hazlewood,246,-36.338,-14.772,3.745,-4.017,11.968
M Prasidh Krishna,330,-95.136,-28.829,3.436,-3.888,11.337
Harpreet Brar,120,-18.492,-15.410,2.766,-0.177,6.281
VG Arora,231,1.209,0.524,2.237,-4.403,9.537
MA Starc,216,10.755,4.979,2.060,-7.630,14.183
